## **Python Memory Management**

Memory management in Python involves a combination of automatic garbage collection, reference counting, and various internal optimizations to efficiently manage memory allocation and deallocation. Understanding these mechanisms can help developers write more efficient and robust applications.

1. Key Concepts in Python Memory Management
2. Memory Allocation and Deallocation
3. Reference Counting
4. Garbage Collection
5. The gc Module
6. Memory Management Best Practices

### Reference Counting
Reference counting is the primary method Python uses to manage memory. Each object in Python maintains a count of references pointing to it. When the reference count drops to zero, the memory occupied by the object is deallocated.

In [2]:
import sys

a=[]
## 2 (one reference from 'a' and one from getrefcount()) 
print(sys.getrefcount(a))

2


In [4]:
b=a
## 3 (a is one reference, b is another and the third one is getrefcount())
print(sys.getrefcount(b))

3


In [ ]:
del b
## 2 (because b is deleted)
print(sys.getrefcount(a))

2


so when the reference drops to 0, memory occupied by that object will be deallocated.

#### Garbage Collection (in other programming lang we use destructor but in python there is garbage collector)
Python includes a cyclic garbage collector to handle reference cycles. Reference cycles occur when objects reference each other, preventing their reference counts from reaching zero.

In [6]:
import gc
## enabling garbage collection
gc.enable()

In [7]:
## disabling garbage collection
gc.disable()

In [9]:
## Manually trigger garbage collection
gc.collect() ## unreachable objects are returned (e.g., 3048)

0

In [11]:
## Get garbage collection stats
print(gc.get_stats()) ## Returns list of dictionaries containing per-generation statistics.

[{'collections': 0, 'collected': 0, 'uncollectable': 0}, {'collections': 12, 'collected': 498, 'uncollectable': 0}, {'collections': 2, 'collected': 3048, 'uncollectable': 0}]


In [13]:
## Get unreachable objects
print(gc.garbage)

[]


### **Memory Management Best Practices**

1.  **Use Local Variables:** Local variables have a shorter lifespan and are freed sooner than global variables.
2.  **Avoid Circular References:** Circular references can lead to memory leaks if not properly managed.
3.  **Use Generators:** Generators produce items one at a time and only keep one item in memory at a time, making them memory efficient.
4.  **Explicitly Delete Objects:** Use the `del` statement to delete variables and objects explicitly.
5.  **Profile Memory Usage:** Use memory profiling tools like `tracemalloc` and `memory_profiler` to identify memory leaks and optimize memory usage.

---

### **Why These Matter**

To put these into context, especially the more technical points:

* **Generators vs. Lists:** If you're processing a million rows of data, a list loads all of them into your RAM at once (which might crash your app). A generator acts like a "conveyor belt," bringing you one item at a time so your memory usage stays flat.
    
* **The Power of `tracemalloc`:** This is a built-in library that acts like a detective. It can tell you exactly which line of code is responsible for allocating the most memory, helping you find those hidden "leaks" caused by circular references.

* **The `del` Statement:** While Python is great at cleaning up after itself, using `del` is like proactively taking out the trash rather than waiting for the garbage truck to arrive. It’s particularly useful for large datasets or image processing buffers.


In [ ]:
## Handling Circular Reference
## Example
import gc

class MyObject:
    def __init__(self,name):
        self.name = name
        print(f"Object {self.name} created")

    def __del__(self):
        print(f"Object {self.name} deleted")

## Creating a circular reference (bad practice, leads to memory leaks)
obj1 = MyObject("obj1")
obj2 = MyObject("obj2")
obj1.ref = obj2
obj2.ref = obj1

## Deleting objects
del obj1
del obj2

## Here the garbage will not be collected because we have used circular reference
## So we need to manually treat garbage collection

## Manually Trigger Garbage Collection (because we used bad practice, circular reference)
gc.collect()

Object obj1 created
Object obj2 created
Object obj1 deleted
Object obj2 deleted


9

#### Gererators For Memory Efficiency
Generators allow you to produce items one at a time, using memory efficiently by only keeping one item in memory at a time.

##### yield keyword
- In Python, the yield keyword is used to create generators, which are a special type of iterator that produces values "lazily"—one at a time, on-demand—rather than all at once.

- The yield keyword turns a function into a function generator. The function generator returns an iterator.The code inside the function is not executed when they are first called, but are divided into steps, one step for each yield, and each step is only executed when iterated upon.

- Unlike the return keyword which stops further execution of the function, the yield keyword returns the result so far, and continues to the next step.

- The return value will be a list of values, one item for each yield.

In [ ]:
def generate_numbers(n):
    for i in range(n):
        yield i

## using the generator
for num in generate_numbers(100000):
    print(num)
    if num>10:
        break


0
1
2
3
4
5
6
7
8
9
10
11


##### Profiling Memory Usage with tracemalloc

In [24]:
import tracemalloc

def create_list():
    return [i for i in range(10000)]

def main():
    tracemalloc.start()

    create_list()

    snapshot = tracemalloc.take_snapshot()
    top_stats = snapshot.statistics('lineno')

    print("[ Top 10 ]")
    for stat in top_stats[:10]:
        print(stat)
    
    print("all")
    for stat in top_stats[::]:
        print(stat)

In [25]:
main()

[ Top 10 ]
c:\Users\Administrator\Desktop\Cloud AI Backend\AI ML DataScience Practice\AI-ML-DataScience-Practice\.venv\Lib\inspect.py:2178: size=304 KiB, count=11, average=27.7 KiB
c:\Users\Administrator\Desktop\Cloud AI Backend\AI ML DataScience Practice\AI-ML-DataScience-Practice\.venv\Lib\selectors.py:305: size=144 KiB, count=3, average=48.0 KiB
c:\Users\Administrator\Desktop\Cloud AI Backend\AI ML DataScience Practice\AI-ML-DataScience-Practice\.venv\Lib\site-packages\IPython\core\compilerop.py:174: size=19.2 KiB, count=203, average=97 B
c:\Users\Administrator\Desktop\Cloud AI Backend\AI ML DataScience Practice\AI-ML-DataScience-Practice\.venv\Lib\site-packages\pygments\formatters\html.py:514: size=18.1 KiB, count=228, average=82 B
c:\Users\Administrator\Desktop\Cloud AI Backend\AI ML DataScience Practice\AI-ML-DataScience-Practice\.venv\Lib\inspect.py:2199: size=13.8 KiB, count=40, average=353 B
c:\Users\Administrator\Desktop\Cloud AI Backend\AI ML DataScience Practice\AI-ML-DataS